<a href="https://colab.research.google.com/github/24dhanush/GUVI_PROJECT/blob/https%2Fgithub.com%2F24dhanush%2FGUVI_PROJECT%2Ftree%2F35b6779e8f953d226911179405ec9dadac6aede8%2Fproject%25205/movie_recom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# loading the data from the csv file to pandas dataframe
movies_data = pd.read_csv('/content/imdb_movies_2024.csv')

In [ ]:
# printing the first 5 rows of the dataframe
movies_data.head()

In [ ]:
# number of rows and columns in the data frame

movies_data.shape

In [ ]:
storline_data=movies_data['Storyline']

In [ ]:
# converting the text data to feature vectors

vectorizer = TfidfVectorizer()

In [ ]:
feature_vectors = vectorizer.fit_transform(storline_data)

In [ ]:
print(feature_vectors)

In [ ]:
# getting the similarity scores using cosine similarity

similarity = cosine_similarity(feature_vectors)

In [ ]:
print(similarity)

In [ ]:
print(similarity.shape)

In [ ]:
# getting the movie storyline from the user

storyline = input(' Enter your favourite movie storyline : ')

In [ ]:
# creating a list with all the movie names given in the dataset

list_of_all_titles = movies_data['Movie Name'].tolist()
print(list_of_all_titles)

In [ ]:
# Vectorize the input storyline
input_storyline_vector = vectorizer.transform([storyline])

# Calculate cosine similarity between input storyline and all movie storylines
similarity_score_with_input = cosine_similarity(input_storyline_vector, feature_vectors)

# Get a list of (index, similarity score) tuples
similarity_scores = list(enumerate(similarity_score_with_input[0]))

# Sort the movies based on their similarity score with the input storyline
sorted_similar_movies = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

print('Movies suggested based on your storyline:\n')

i = 1
for movie_index, score in sorted_similar_movies:
    if i <= 10:
        title = movies_data.iloc[movie_index]['Movie Name']
        print(f'{i}. {title} (Similarity: {score:.4f})')
        i += 1
    else:
        break

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data for plotting (top 10 similar movies)
plot_data = []
for movie_index, score in sorted_similar_movies:
    title = movies_data.iloc[movie_index]['Movie Name']
    plot_data.append({'Movie Name': title, 'Similarity Score': score})

plot_df = pd.DataFrame(plot_data).head(10)

# Create the bar chart
plt.figure(figsize=(12, 7))
sns.barplot(x='Similarity Score', y='Movie Name', data=plot_df, palette='viridis')
plt.title('Top 10 Movie Recommendations by Storyline Similarity')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Movie Name')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
pip install streamlit


In [ ]:
%%writefile movieapp.py
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(layout="wide") # Use wide layout

st.title('🎬 Movie Recommendation System')l


movies_data = pd.read_csv('/content/imdb_movies_2024.csv')

st.markdown("---")

# Data Overview - placed in an expander
with st.expander("📚 View Original Movie Data Sample"):
    st.markdown("A glimpse of the dataset used for recommendations.")
    st.dataframe(movies_data.head(10), use_container_width=True) # Use container width

st.markdown("---")

storline_data = movies_data['Storyline']


vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(storline_data)


similarity = cosine_similarity(feature_vectors)

st.header('🔍 Find Your Next Movie') # Better header for input section
storyline = st.text_input('Enter a storyline or a description of a movie you enjoyed:', '')

if storyline:
    st.subheader('🌟 Top Movie Recommendations Based on Your Input:')


    input_storyline_vector = vectorizer.transform([storyline])


    similarity_score_with_input = cosine_similarity(input_storyline_vector, feature_vectors)


    similarity_scores = list(enumerate(similarity_score_with_input[0]))


    sorted_similar_movies = sorted(similarity_scores, key=lambda x: x[1], reverse=True)


    plot_data = []
    top_recommendations = []
    for i, (movie_index, score) in enumerate(sorted_similar_movies):
        if i < 10:
            title = movies_data.iloc[movie_index]['Movie Name']
            full_storyline = movies_data.iloc[movie_index]['Storyline'] # Get full storyline
            plot_data.append({'Rank': i + 1, 'Movie Name': title, 'Similarity Score': score})
            top_recommendations.append({'Rank': i + 1, 'Movie Name': title, 'Storyline': full_storyline, 'Similarity Score': score})
        else:
            break

    if top_recommendations:
        col1, col2 = st.columns([2, 3])

        with col1:
            st.markdown("### Recommended Movies (Details)")
            for rec in top_recommendations:
                st.markdown(f"---")
                st.markdown(f"**{rec['Rank']}. {rec['Movie Name']}**")
                st.markdown(f"Similarity: `{rec['Similarity Score']:.4f}`")
                st.info(f"*{rec['Storyline']}*")
            st.markdown("---")

        with col2:
            st.markdown("### Visual Summary")
            plot_df = pd.DataFrame(plot_data)


            fig = plt.figure(figsize=(10, 6))
            sns.barplot(x='Similarity Score', y='Movie Name', data=plot_df.sort_values(by='Similarity Score', ascending=False), palette='viridis', hue='Movie Name', legend=False)
            plt.title('Top 10 Movie Recommendations by Storyline Similarity', fontsize=14)
            plt.xlabel('Cosine Similarity Score', fontsize=10)
            plt.ylabel('Movie Name', fontsize=10)
            plt.xlim(0, 1.05)
            plt.xticks(fontsize=8)
            plt.yticks(fontsize=8)
            plt.grid(axis='x', linestyle='--', alpha=0.7)
            plt.tight_layout()
            st.pyplot(fig, use_container_width=True)
    else:
        st.warning("No similar movies found for your storyline. Please try a different input.")


st.sidebar.markdown("---")
st.sidebar.write("Developed by DHANUSH DB")

In [ ]:
!streamlit run movieapp.py & npx localtunnel --port 8501